# Klasifikasi Telur — Training MobileNetV1 α=0.25
**Tugas Akhir — Politeknik Negeri Sriwijaya**  
Muhammad Reka Alviandi — NIM 062330701499

**Output:** `egg_model.h` → di-copy ke folder sketch Arduino

---
### Urutan sel:
1. Setup & mount Drive
2. Upload dataset
3. Cek dataset
4. Pipeline augmentasi
5. Bangun model
6. Fase 1 — Train head
7. Fase 2 — Fine-tune
8. Plot training
9. Konversi TFLite INT8
10. Generate `egg_model.h`
11. Evaluasi & download

## 0. Cek GPU

In [ ]:
import tensorflow as tf
import os

print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU        : {gpus[0].name}  ✓')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('GPU        : tidak ada — ganti Runtime → T4 GPU')

## 1. Mount Google Drive (opsional)
Skip sel ini jika mau upload langsung tanpa Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan path ini dengan lokasi dataset di Drive kamu
DRIVE_DATASET = '/content/drive/MyDrive/KlasifikasiTelur/dataset'
print(f'Drive dataset path: {DRIVE_DATASET}')
print('Ada:', len(os.listdir(DRIVE_DATASET)), 'file' if os.path.isdir(DRIVE_DATASET) else '— folder tidak ditemukan')

## 2. Upload Dataset
**Pilih salah satu cara:**
- **Cara A**: Upload ZIP dari komputer lokal
- **Cara B**: Salin dari Google Drive (jalankan sel Drive dulu)

Format file: `good_0001.jpg`, `bad_0001.jpg` dst (prefix = label)

In [ ]:
# ── CARA A: Upload ZIP ────────────────────────────────────────
# Buat ZIP dulu: klik kanan folder dataset → Send to → Compressed
# Isi ZIP: langsung file .jpg (bukan subfolder)

from google.colab import files
import zipfile, pathlib

DATASET_DIR = pathlib.Path('/content/dataset')
DATASET_DIR.mkdir(exist_ok=True)

print('Pilih file ZIP dataset...')
uploaded = files.upload()   # dialog pilih file

for fname in uploaded:
    zip_path = pathlib.Path(fname)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for member in z.namelist():
            # ekstrak hanya file .jpg/.jpeg/.png, abaikan subfolder
            basename = pathlib.Path(member).name
            if basename.lower().endswith(('.jpg', '.jpeg', '.png')) and basename:
                dest = DATASET_DIR / basename
                dest.write_bytes(z.read(member))
    print(f'ZIP "{fname}" diekstrak ke {DATASET_DIR}')

print(f'Total file: {len(list(DATASET_DIR.iterdir()))}')

In [ ]:
# ── CARA B: Salin dari Drive (jalankan jika sudah mount Drive) ──
import shutil, pathlib

DATASET_DIR  = pathlib.Path('/content/dataset')
DATASET_DIR.mkdir(exist_ok=True)
DRIVE_DATASET = '/content/drive/MyDrive/KlasifikasiTelur/dataset'  # sesuaikan

for f in pathlib.Path(DRIVE_DATASET).iterdir():
    if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
        shutil.copy(f, DATASET_DIR / f.name)

print(f'Disalin {len(list(DATASET_DIR.iterdir()))} file dari Drive')

## 3. Cek Dataset

In [ ]:
import pathlib, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATASET_DIR = pathlib.Path('/content/dataset')
MODEL_DIR   = pathlib.Path('/content/models')
MODEL_DIR.mkdir(exist_ok=True)

all_files, all_labels = [], []
for f in sorted(DATASET_DIR.iterdir()):
    if f.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
        continue
    name = f.name.lower()
    if   name.startswith('good'): label = 1
    elif name.startswith('bad'):  label = 0
    else:
        print(f'[SKIP] Nama tidak dikenal: {f.name}')
        continue
    all_files.append(str(f))
    all_labels.append(label)

n_good = sum(all_labels)
n_bad  = len(all_labels) - n_good
print(f'BAGUS      : {n_good} gambar')
print(f'TIDAK BAGUS: {n_bad}  gambar')
print(f'Total      : {len(all_files)} gambar')

if n_good < 10 or n_bad < 10:
    print('\n[PERINGATAN] Idealnya minimal 100 gambar per kelas.')
    print('Dataset kecil akan dipakai augmentasi berat.')

# Preview beberapa gambar
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
good_files = [f for f, l in zip(all_files, all_labels) if l == 1][:5]
bad_files  = [f for f, l in zip(all_files, all_labels) if l == 0][:5]
for i, path in enumerate(good_files):
    axes[0][i].imshow(Image.open(path).resize((96, 96)))
    axes[0][i].set_title('BAGUS', fontsize=9)
    axes[0][i].axis('off')
for i, path in enumerate(bad_files):
    axes[1][i].imshow(Image.open(path).resize((96, 96)))
    axes[1][i].set_title('TIDAK BAGUS', fontsize=9)
    axes[1][i].axis('off')
plt.suptitle('Preview Dataset (96×96)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Konfigurasi & Pipeline Augmentasi

In [ ]:
import tensorflow as tf

# ── Konfigurasi ──────────────────────────────────────────────
IMG_SIZE   = 96
ALPHA      = 0.25
BATCH_SIZE = 16    # Colab GPU bisa lebih besar dari lokal
EPOCHS_1   = 60
EPOCHS_2   = 40
REPEAT     = 30    # augmentasi: tiap epoch = 30× data asli

# ── Split train/val 80/20 stratified ────────────────────────
rng      = np.random.default_rng(42)
idx_good = [i for i, l in enumerate(all_labels) if l == 1]
idx_bad  = [i for i, l in enumerate(all_labels) if l == 0]

def split_indices(idx, val_ratio=0.2):
    idx = rng.permutation(idx).tolist()
    n_val = max(1, round(len(idx) * val_ratio))
    return idx[n_val:], idx[:n_val]

tr_good, va_good = split_indices(idx_good)
tr_bad,  va_bad  = split_indices(idx_bad)
train_idx = tr_good + tr_bad
val_idx   = va_good + va_bad
rng.shuffle(train_idx)

train_files  = [all_files[i]  for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]
val_files    = [all_files[i]  for i in val_idx]
val_labels   = [all_labels[i] for i in val_idx]

print(f'Train : {len(train_files)} gambar  (GOOD={sum(train_labels)}, BAD={len(train_labels)-sum(train_labels)})')
print(f'Val   : {len(val_files)} gambar  (GOOD={sum(val_labels)}, BAD={len(val_labels)-sum(val_labels)})')

# ── Fungsi load & augmentasi ────────────────────────────────
def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.cast(label, tf.float32)

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.4)
    img = tf.image.random_contrast(img, lower=0.6, upper=1.4)
    img = tf.image.random_saturation(img, lower=0.6, upper=1.4)
    img = tf.image.random_hue(img, max_delta=0.1)
    # crop-resize simulasi rotasi kecil
    img = tf.image.random_crop(
        tf.pad(img, [[8,8],[8,8],[0,0]], constant_values=0.5),
        [IMG_SIZE, IMG_SIZE, 3]
    )
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(train_files), tf.constant(train_labels, dtype=tf.int32))
    )
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .cache()
    .repeat()             # infinite — steps_per_epoch mengontrol panjang epoch
    .map(augment, num_parallel_calls=AUTOTUNE)
    .shuffle(1024)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices(
        (tf.constant(val_files), tf.constant(val_labels, dtype=tf.int32))
    )
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

steps_per_epoch = max(1, (len(train_files) * REPEAT) // BATCH_SIZE)
print(f'\nAugmentasi × {REPEAT} → ~{len(train_files)*REPEAT} sampel/epoch')
print(f'Steps per epoch : {steps_per_epoch}')

## 5. Bangun Model MobileNetV1 α=0.25

In [ ]:
base = tf.keras.applications.MobileNet(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    alpha=ALPHA,
    include_top=False,
    weights='imagenet',
)
base.trainable = False

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dropout(0.5)(x)
x       = tf.keras.layers.Dense(64, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)

# Hitung parameter
total   = model.count_params()
trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'Total params    : {total:,}')
print(f'Trainable (head): {trainable:,}')
model.summary(line_length=65)

## 6. Fase 1 — Train Head (base frozen)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

callbacks1 = [
    tf.keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / 'best_phase1.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=20,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10,
        min_lr=1e-6, verbose=1
    ),
]

print(f'Fase 1: {EPOCHS_1} epoch, head only')
hist1 = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS_1,
    validation_data=val_ds,
    callbacks=callbacks1,
)

best_val1 = max(hist1.history.get('val_accuracy', [0]))
print(f'\nBest val accuracy fase 1: {best_val1*100:.1f}%')

## 7. Fase 2 — Fine-tune Top 20 Layers

In [ ]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

trainable2 = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'Trainable params fase 2: {trainable2:,}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

callbacks2 = [
    tf.keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / 'best_phase2.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=15,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=8,
        min_lr=1e-7, verbose=1
    ),
]

print(f'Fase 2: {EPOCHS_2} epoch, fine-tune top 20 layers')
hist2 = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS_2,
    validation_data=val_ds,
    callbacks=callbacks2,
)

best_val2  = max(hist2.history.get('val_accuracy', [0]))
final_val  = max(best_val1, best_val2)
print(f'Best val accuracy fase 2: {best_val2*100:.1f}%')
print(f'Final best val accuracy : {final_val*100:.1f}%')

## 8. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

acc1  = hist1.history['accuracy']
vacc1 = hist1.history['val_accuracy']
acc2  = hist2.history['accuracy']
vacc2 = hist2.history['val_accuracy']

acc_all  = acc1 + acc2
vacc_all = vacc1 + vacc2
ep_split = len(acc1)
ep_all   = list(range(1, len(acc_all) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ep_all, acc_all,  label='Train acc')
ax1.plot(ep_all, vacc_all, label='Val acc')
ax1.axvline(ep_split, color='gray', linestyle='--', label='Fine-tune mulai')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

loss1  = hist1.history['loss']
vloss1 = hist1.history['val_loss']
loss2  = hist2.history['loss']
vloss2 = hist2.history['val_loss']
loss_all  = loss1 + loss2
vloss_all = vloss1 + vloss2

ax2.plot(ep_all, loss_all,  label='Train loss')
ax2.plot(ep_all, vloss_all, label='Val loss')
ax2.axvline(ep_split, color='gray', linestyle='--', label='Fine-tune mulai')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Training — Best val acc: {final_val*100:.1f}%', fontsize=13)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'training_history.png'), dpi=120)
plt.show()
print('Plot disimpan ke models/training_history.png')

## 9. Konversi TFLite INT8
Quantisasi penuh (full integer) — wajib untuk TFLite Micro di ESP32-S3

In [ ]:
import pathlib

MODEL_DIR = pathlib.Path('/content/models')

# Simpan model Keras
keras_path = str(MODEL_DIR / 'egg_classifier.keras')
model.save(keras_path)
print(f'Keras model: {keras_path}')

# ── Float32 TFLite ─────────────────────────────────────────
conv_f   = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_f = conv_f.convert()
path_f32 = MODEL_DIR / 'egg_classifier_float32.tflite'
path_f32.write_bytes(tflite_f)
print(f'Float32 TFLite : {path_f32}  ({len(tflite_f)/1024:.1f} KB)')

# ── INT8 Full Quantization ─────────────────────────────────
def representative_dataset():
    for path in all_files:
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        yield [tf.expand_dims(img, 0)]

conv_q = tf.lite.TFLiteConverter.from_keras_model(model)
conv_q.optimizations             = [tf.lite.Optimize.DEFAULT]
conv_q.representative_dataset    = representative_dataset
conv_q.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv_q.inference_input_type      = tf.int8
conv_q.inference_output_type     = tf.int8

print('\nMengkonversi ke INT8 (proses beberapa detik)...')
tflite_q = conv_q.convert()
path_int8 = MODEL_DIR / 'egg_classifier_int8.tflite'
path_int8.write_bytes(tflite_q)
print(f'INT8 TFLite    : {path_int8}  ({len(tflite_q)/1024:.1f} KB)')
print(f'\nKompresi: {len(tflite_f)/1024:.1f} KB → {len(tflite_q)/1024:.1f} KB  ({(1-len(tflite_q)/len(tflite_f))*100:.0f}% lebih kecil)')

## 10. Generate `egg_model.h`
File ini di-copy ke folder sketch Arduino (`EggClassifierV2/`)

In [ ]:
model_data = path_int8.read_bytes()

# Format hex array, 12 byte per baris
hex_lines = []
for i in range(0, len(model_data), 12):
    chunk = model_data[i:i+12]
    hex_lines.append('  ' + ', '.join(f'0x{b:02x}' for b in chunk))

header = f"""// Auto-generated — JANGAN EDIT MANUAL
// Model  : MobileNetV1 \u03b1={ALPHA}, input 96\u00d796 INT8
// Val acc: {final_val*100:.1f}%
// Size   : {len(model_data)} bytes ({len(model_data)/1024:.1f} KB)

#pragma once
#include <stdint.h>

const unsigned int g_model_len = {len(model_data)};

alignas(8) const uint8_t g_model_data[] = {{
{chr(10).join(hex_lines)}
}};
"""

header_path = MODEL_DIR / 'egg_model.h'
header_path.write_text(header)

print(f'Header   : {header_path}')
print(f'Array    : g_model_data[{len(model_data)}]  ({len(model_data)/1024:.1f} KB)')
print(f'Val acc  : {final_val*100:.1f}%')
print()
print('─' * 50)
print('LANGKAH SELANJUTNYA:')
print('1. Download egg_model.h (sel berikutnya)')
print('2. Copy ke folder EggClassifierV2/')
print('3. Compile & upload ke ESP32-S3')
print('─' * 50)

## 11. Evaluasi & Download File

In [ ]:
# ── Evaluasi model di validation set ──────────────────────
loss, acc = model.evaluate(val_ds, verbose=0)
print('='*50)
print('EVALUASI AKHIR (validation set)')
print('='*50)
print(f'Val Loss     : {loss:.4f}')
print(f'Val Accuracy : {acc*100:.1f}%')

# ── Confusion matrix ───────────────────────────────────────
import numpy as np

y_true, y_pred = [], []
for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0).flatten()
    y_true.extend(labels.numpy())
    y_pred.extend((preds >= 0.5).astype(int))

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_true, y_pred)
print()
print('Confusion Matrix:')
print(f'              Pred BAD  Pred GOOD')
print(f'  True BAD  :   {cm[0][0]:5d}      {cm[0][1]:5d}')
print(f'  True GOOD :   {cm[1][0]:5d}      {cm[1][1]:5d}')
print()
print(classification_report(y_true, y_pred, target_names=['BAD', 'GOOD']))

if acc < 0.85:
    print('[PERINGATAN] Akurasi < 85% — tambah dataset (target 100+ per kelas)')

In [ ]:
# ── Verifikasi model INT8 dengan TFLite interpreter ────────
import numpy as np
from PIL import Image

interp = tf.lite.Interpreter(model_path=str(path_int8))
interp.allocate_tensors()

inp_detail = interp.get_input_details()[0]
out_detail = interp.get_output_details()[0]
in_scale   = inp_detail['quantization'][0]
in_zp      = inp_detail['quantization'][1]
out_scale  = out_detail['quantization'][0]
out_zp     = out_detail['quantization'][1]

print('INT8 Model details:')
print(f'  Input  shape : {inp_detail["shape"]}  dtype={inp_detail["dtype"].__name__}')
print(f'  Output shape : {out_detail["shape"]}  dtype={out_detail["dtype"].__name__}')
print(f'  In  quant    : scale={in_scale:.6f}  zero_point={in_zp}')
print(f'  Out quant    : scale={out_scale:.6f}  zero_point={out_zp}')

# Test satu gambar
test_path = all_files[0]
test_label = all_labels[0]
img = np.array(Image.open(test_path).resize((96, 96)), dtype=np.float32) / 255.0
img_q = np.clip(np.round(img / in_scale) + in_zp, -128, 127).astype(np.int8)
interp.set_tensor(inp_detail['index'], img_q[np.newaxis])
interp.invoke()
out_q = interp.get_tensor(out_detail['index'])[0][0]
score = (int(out_q) - out_zp) * out_scale
pred  = 'BAGUS' if score >= 0.5 else 'TIDAK BAGUS'
truth = 'BAGUS' if test_label == 1 else 'TIDAK BAGUS'
print(f'\nTest gambar: {test_path}')
print(f'  Prediksi : {pred}  (score={score:.3f})')
print(f'  Label    : {truth}  ← {"✓ BENAR" if pred == truth else "✗ SALAH"}')

In [ ]:
# ── Download file hasil ─────────────────────────────────────
from google.colab import files
import pathlib

MODEL_DIR = pathlib.Path('/content/models')

print('Mengunduh egg_model.h ...')
files.download(str(MODEL_DIR / 'egg_model.h'))

print('Mengunduh egg_classifier_int8.tflite ...')
files.download(str(MODEL_DIR / 'egg_classifier_int8.tflite'))

print('Mengunduh training_history.png ...')
files.download(str(MODEL_DIR / 'training_history.png'))

print('\nSelesai! Salin egg_model.h ke folder sketch Arduino.')

## (Opsional) Simpan semua ke Google Drive

In [ ]:
import shutil, pathlib

DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/KlasifikasiTelur/models')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

for f in pathlib.Path('/content/models').iterdir():
    shutil.copy(f, DRIVE_OUT / f.name)
    print(f'  Disimpan: {f.name}')

print(f'\nSemua model tersimpan di Drive: {DRIVE_OUT}')